# Bee-Wo Baseline Training on Google Colab

This notebook mounts Google Drive, copies the DL project files into Colab, unzips `data_clean.zip`, runs a smoke test, runs one selected baseline model end-to-end, and copies the results back to Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
# Colab config
from pathlib import Path

PROJECT_DIR_IN_DRIVE = '/content/drive/MyDrive/dl_project'
DATA_ZIP_IN_DRIVE = '/content/drive/MyDrive/dl_project/data_clean.zip'
RESULTS_DIR_IN_DRIVE = '/content/drive/MyDrive/dl_project/colab_runs'

MODEL_NAME = 'resnet10_landmark_fusion'  # simple_cnn | resnet10_3d | mobilenetv2_3d | cnn_lstm | resnet10_temporal_attention | resnet10_landmark_fusion | r2plus1d_18 | mediapipe_logreg | mediapipe_random_forest
EPOCHS = 30
BATCH_SIZE = 4
IMAGE_SIZE = 64
SEQ_LEN = 16
SEED = 20260331

WORKDIR = Path('/content/dl_project')
WORKDIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = Path(RESULTS_DIR_IN_DRIVE)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print('Project dir in Drive:', PROJECT_DIR_IN_DRIVE)
print('Dataset zip in Drive:', DATA_ZIP_IN_DRIVE)
print('Model:', MODEL_NAME)


Project dir in Drive: /content/drive/MyDrive/dl_project
Dataset zip in Drive: /content/drive/MyDrive/dl_project/data_clean.zip
Model: resnet10_landmark_fusion


In [19]:
import shutil
from pathlib import Path

drive_project = Path(PROJECT_DIR_IN_DRIVE)
assert drive_project.exists(), f'Missing project directory: {drive_project}'
assert Path(DATA_ZIP_IN_DRIVE).exists(), f'Missing dataset zip: {DATA_ZIP_IN_DRIVE}'

files_to_copy = [
    'train_baseline.py',
    'bee_wo_dataset.py',
    'data_clean/annotations.csv',
    'data_clean/splits.csv',
    'data_clean/label_map.csv',
    'baseline_models/__init__.py',
    'baseline_models/mediapipe_baseline.py',
    'baseline_models/resnet10_3d.py',
    'baseline_models/mobilenetv2_3d.py',
    'project_models/__init__.py',
    'project_models/r2plus1d_18.py',
    'project_models/cnn_lstm.py',
    'project_models/resnet10_temporal_attention.py',
    'project_models/resnet10_landmark_fusion.py',
    'train_mediapipe_baseline.py',
    'scripts/extract_mediapipe_landmarks.py',
]

for relative_path in files_to_copy:
    src = drive_project / relative_path
    dst = WORKDIR / relative_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

shutil.copy2(DATA_ZIP_IN_DRIVE, WORKDIR / 'data_clean.zip')
print('Copied project files into', WORKDIR)

Copied project files into /content/dl_project


In [5]:
%cd /content/dl_project
!python -m pip install --quiet --upgrade pip
!python -m pip install --quiet torch torchvision pillow scikit-learn numpy

/content/dl_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.2 MB/s eta 0:00:00


In [6]:
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.14


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 47.2 MB/s  0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mediapipe]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.


In [7]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Selected device:', 'cuda' if torch.cuda.is_available() else 'cpu')

Torch version: 2.10.0+cu128
CUDA available: True
Selected device: cuda


In [8]:
import zipfile
from pathlib import Path

zip_path = Path('/content/dl_project/data_clean.zip')
extract_root = Path('/content/dl_project')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_root)

all_dir = extract_root / 'data_clean' / 'all'
assert all_dir.exists(), f'Missing extracted dataset directory: {all_dir}'
print('Extracted data_clean.zip to', extract_root)
print('Gesture folders:', sorted(p.name for p in all_dir.iterdir() if p.is_dir()))

Extracted data_clean.zip to /content/dl_project
Gesture folders: ['G01', 'G02', 'G05', 'G06', 'G07']


In [9]:
!python - <<'PY'
from pathlib import Path
from bee_wo_dataset import BeeWoClipDataset

for split in ['train', 'val', 'test']:
    ds = BeeWoClipDataset(
        Path('data_clean'),
        Path('data_clean/annotations.csv'),
        Path('data_clean/splits.csv'),
        Path('data_clean/label_map.csv'),
        split=split,
        seq_len=16,
        image_size=64,
    )
    print(split, len(ds), tuple(ds[0]['frames'].shape))

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
train 700 (16, 3, 64, 64)
val 150 (16, 3, 64, 64)
test 150 (16, 3, 64, 64)


## Landmark Extraction

This only runs for `resnet10_landmark_fusion`. Other models skip this step.

In [10]:
import shutil
from pathlib import Path

LANDMARK_MODELS = {'resnet10_landmark_fusion', 'mediapipe_logreg', 'mediapipe_random_forest'}
runtime_landmarks = Path('/content/dl_project/data_clean/landmarks')
drive_landmarks = Path('/content/drive/MyDrive/dl_project/data_clean/landmarks')

if MODEL_NAME in LANDMARK_MODELS:
    if drive_landmarks.exists() and any(drive_landmarks.glob('*.npy')):
        if runtime_landmarks.exists():
            shutil.rmtree(runtime_landmarks)
        shutil.copytree(drive_landmarks, runtime_landmarks)
        print(f'Restored cached landmarks from Drive: {drive_landmarks}')
        print(f'Runtime landmark files: {len(list(runtime_landmarks.glob("*.npy")))}')
    else:
        print('No cached Drive landmarks found; extracting with MediaPipe now.')
        !python -u scripts/extract_mediapipe_landmarks.py --seq-len {SEQ_LEN} --log-every 25

        assert runtime_landmarks.exists(), f'Landmark extraction did not create: {runtime_landmarks}'
        assert any(runtime_landmarks.glob('*.npy')), f'No landmark .npy files found in: {runtime_landmarks}'

        drive_landmarks.parent.mkdir(parents=True, exist_ok=True)
        if drive_landmarks.exists():
            shutil.rmtree(drive_landmarks)
        shutil.copytree(runtime_landmarks, drive_landmarks)
        print(f'Saved extracted landmarks to Drive: {drive_landmarks}')
else:
    print(f'Skipping landmark restore/extraction for model: {MODEL_NAME}')

Restored cached landmarks from Drive: /content/drive/MyDrive/dl_project/data_clean/landmarks
Runtime landmark files: 1000


In [28]:
#if MODEL_NAME == 'resnet10_landmark_fusion':
 #   !python -u scripts/extract_mediapipe_landmarks.py --seq-len {SEQ_LEN}
#else:
 #   print(f'Skipping landmark extraction for model: {MODEL_NAME}')


2026-04-07 21:27:44.741579: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-07 21:27:44.760502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775597264.783968   12188 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775597264.791411   12188 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775597264.810612   12188 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [29]:
#import shutil
#from pathlib import Path

#src = Path("/content/dl_project/data_clean/landmarks")
#dst = Path("/content/drive/MyDrive/dl_project/data_clean/landmarks")

#if dst.exists():
 #   shutil.rmtree(dst)
#shutil.copytree(src, dst)

#print("Copied landmarks to Drive:", dst)


Copied landmarks to Drive: /content/drive/MyDrive/dl_project/data_clean/landmarks


## Smoke Test

Run this once before the full training job. It verifies imports, data loading, batching, checkpoint writing, and metrics generation.

In [16]:
if MODEL_NAME == 'resnet10_landmark_fusion':
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs 1 \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED} \
      --landmarks-root data_clean/landmarks \
      --max-train-batches 1 \
      --max-eval-batches 1

elif MODEL_NAME == 'mediapipe_logreg':
    !python -u train_mediapipe_baseline.py \
      --classifier logreg \
      --seq-len {SEQ_LEN} \
      --landmarks-root data_clean/landmarks \
      --max-train-samples 10

elif MODEL_NAME == 'mediapipe_random_forest':
    !python -u train_mediapipe_baseline.py \
      --classifier random_forest \
      --seq-len {SEQ_LEN} \
      --landmarks-root data_clean/landmarks \
      --max-train-samples 10

else:
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs 1 \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED} \
      --max-train-batches 1 \
      --max-eval-batches 1


{"epoch": 1, "train_loss": 1.585984468460083, "val_loss": 1.613248586654663, "train_accuracy": 0.25, "train_macro_f1": 0.16666666666666666, "val_accuracy": 0.25, "val_macro_f1": 0.1}
Saved run artifacts to runs/resnet10_landmark_fusion_20260412_214127
{
  "model": "resnet10_landmark_fusion",
  "val_accuracy": 0.25,
  "val_macro_f1": 0.1,
  "test_accuracy": 0.25,
  "test_macro_f1": 0.1
}


## Real Training Run

In [20]:
!python -u train_mediapipe_baseline.py \
  --classifier logreg \
  --seq-len {SEQ_LEN} \
  --landmarks-root data_clean/landmarks


Saved MediaPipe baseline artifacts to runs/mediapipe_logreg_20260412_214836
{
  "model": "mediapipe_baseline",
  "classifier": "logreg",
  "val_accuracy": 0.88,
  "val_macro_f1": 0.8802085502761994,
  "test_accuracy": 0.9,
  "test_macro_f1": 0.8977840878008185
}


In [21]:
!python -u train_mediapipe_baseline.py \
  --classifier random_forest \
  --seq-len {SEQ_LEN} \
  --landmarks-root data_clean/landmarks


Saved MediaPipe baseline artifacts to runs/mediapipe_random_forest_20260412_214925
{
  "model": "mediapipe_baseline",
  "classifier": "random_forest",
  "val_accuracy": 0.86,
  "val_macro_f1": 0.8609546912693993,
  "test_accuracy": 0.8,
  "test_macro_f1": 0.7992543082509252
}


In [22]:
# simple_cnn | resnet10_3d | mobilenetv2_3d | cnn_lstm | resnet10_temporal_attention | resnet10_landmark_fusion | r2plus1d_18 | mediapipe_logreg | mediapipe_random_forest
!python -u train_baseline.py \
  --model resnet10_3d \
  --epochs 30 \
  --batch-size 4 \
  --image-size 64 \
  --seq-len 16 \



{"epoch": 1, "train_loss": 1.763732102257865, "val_loss": 1.6380074167251586, "train_accuracy": 0.20142857142857143, "train_macro_f1": 0.2015265243303194, "val_accuracy": 0.2, "val_macro_f1": 0.09212121212121212}
{"epoch": 2, "train_loss": 1.6916700376783098, "val_loss": 1.6199138975143432, "train_accuracy": 0.20857142857142857, "train_macro_f1": 0.2074353336827211, "val_accuracy": 0.23333333333333334, "val_macro_f1": 0.1326315789473684}
{"epoch": 3, "train_loss": 1.638575121334621, "val_loss": 1.7842016537984213, "train_accuracy": 0.25857142857142856, "train_macro_f1": 0.2513233932912685, "val_accuracy": 0.21333333333333335, "val_macro_f1": 0.09281385281385282}
{"epoch": 4, "train_loss": 1.5674345758983068, "val_loss": 1.688985382715861, "train_accuracy": 0.29714285714285715, "train_macro_f1": 0.29316914567913255, "val_accuracy": 0.2733333333333333, "val_macro_f1": 0.19268580159336463}
{"epoch": 5, "train_loss": 1.3945410694394793, "val_loss": 1.2458086284001668, "train_accuracy": 0.3

In [ ]:
if MODEL_NAME == 'resnet10_landmark_fusion':
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs {EPOCHS} \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED} \
      --landmarks-root data_clean/landmarks

elif MODEL_NAME == 'mediapipe_logreg':
    !python -u train_mediapipe_baseline.py \
      --classifier logreg \
      --seq-len {SEQ_LEN} \
      --landmarks-root data_clean/landmarks

elif MODEL_NAME == 'mediapipe_random_forest':
    !python -u train_mediapipe_baseline.py \
      --classifier random_forest \
      --seq-len {SEQ_LEN} \
      --landmarks-root data_clean/landmarks

else:
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs {EPOCHS} \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED}


In [30]:
if MODEL_NAME == 'resnet10_landmark_fusion':
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs 30 \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED} \
      --landmarks-root data_clean/landmarks
else:
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs {EPOCHS} \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED}


{"epoch": 1, "train_loss": 1.6020319564001901, "val_loss": 1.5411233854293824, "train_accuracy": 0.22428571428571428, "train_macro_f1": 0.20452504756251805, "val_accuracy": 0.29333333333333333, "val_macro_f1": 0.18561731431382195}
{"epoch": 2, "train_loss": 1.5106622328077044, "val_loss": 1.474538820584615, "train_accuracy": 0.32857142857142857, "train_macro_f1": 0.27818454381414576, "val_accuracy": 0.36, "val_macro_f1": 0.27151681207489126}
{"epoch": 3, "train_loss": 1.4293355407033648, "val_loss": 1.3902053006490072, "train_accuracy": 0.3757142857142857, "train_macro_f1": 0.33661060769666895, "val_accuracy": 0.4, "val_macro_f1": 0.3480016132972616}
{"epoch": 4, "train_loss": 1.350470553466252, "val_loss": 1.384502124786377, "train_accuracy": 0.43857142857142856, "train_macro_f1": 0.41644863104831786, "val_accuracy": 0.42, "val_macro_f1": 0.39981684981684984}
{"epoch": 5, "train_loss": 1.3060600342069353, "val_loss": 1.1952619592348734, "train_accuracy": 0.44, "train_macro_f1": 0.4261

In [20]:
# different learning rate for resnet10_temporal_attention
!python -u train_baseline.py \
  --model resnet10_temporal_attention \
  --epochs 20 \
  --batch-size 4 \
  --image-size 64 \
  --seq-len 16 \
  --learning-rate 3e-4


{"epoch": 1, "train_loss": 1.6458720193590437, "val_loss": 1.6174051253000896, "train_accuracy": 0.18285714285714286, "train_macro_f1": 0.1761003465863068, "val_accuracy": 0.2, "val_macro_f1": 0.06666666666666667}
{"epoch": 2, "train_loss": 1.6215649339130946, "val_loss": 1.6206335576375326, "train_accuracy": 0.18428571428571427, "train_macro_f1": 0.17779970659694763, "val_accuracy": 0.22, "val_macro_f1": 0.12547092547092548}
{"epoch": 3, "train_loss": 1.6251735101427351, "val_loss": 1.609468978246053, "train_accuracy": 0.19285714285714287, "train_macro_f1": 0.18314905363934042, "val_accuracy": 0.2, "val_macro_f1": 0.06666666666666667}
{"epoch": 4, "train_loss": 1.6168299783979143, "val_loss": 1.7014630683263143, "train_accuracy": 0.20285714285714285, "train_macro_f1": 0.19325525236723046, "val_accuracy": 0.2, "val_macro_f1": 0.06666666666666667}
{"epoch": 5, "train_loss": 1.6105627632141113, "val_loss": 1.6054671096801758, "train_accuracy": 0.21285714285714286, "train_macro_f1": 0.209

In [17]:
# resnet10_temporal_attention
if MODEL_NAME == 'resnet10_landmark_fusion':
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs 20 \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED} \
      --landmarks-root data_clean/landmarks
else:
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs {EPOCHS} \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED}


{"epoch": 1, "train_loss": 1.6471615702765328, "val_loss": 1.6215355078379312, "train_accuracy": 0.19428571428571428, "train_macro_f1": 0.18761414751922081, "val_accuracy": 0.2, "val_macro_f1": 0.06666666666666667}
{"epoch": 2, "train_loss": 1.624458804130554, "val_loss": 1.6208651717503866, "train_accuracy": 0.19, "train_macro_f1": 0.18758295517389226, "val_accuracy": 0.2, "val_macro_f1": 0.06666666666666667}
{"epoch": 3, "train_loss": 1.620266238621303, "val_loss": 1.607162888844808, "train_accuracy": 0.20285714285714285, "train_macro_f1": 0.19919374497086656, "val_accuracy": 0.24, "val_macro_f1": 0.1375921375921376}
{"epoch": 4, "train_loss": 1.6150230257851736, "val_loss": 1.6394297901789348, "train_accuracy": 0.20142857142857143, "train_macro_f1": 0.19298722518809577, "val_accuracy": 0.19333333333333333, "val_macro_f1": 0.11417016121517101}
{"epoch": 5, "train_loss": 1.6121151311056954, "val_loss": 1.738130702972412, "train_accuracy": 0.18428571428571427, "train_macro_f1": 0.17910

In [22]:
# resnet10_3d baseline
if MODEL_NAME == 'resnet10_landmark_fusion':
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs 20 \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED} \
      --landmarks-root data_clean/landmarks
else:
    !python -u train_baseline.py \
      --model {MODEL_NAME} \
      --epochs {EPOCHS} \
      --batch-size {BATCH_SIZE} \
      --image-size {IMAGE_SIZE} \
      --seq-len {SEQ_LEN} \
      --seed {SEED}


{"epoch": 1, "train_loss": 1.7707213483537947, "val_loss": 1.6608505614598592, "train_accuracy": 0.21571428571428572, "train_macro_f1": 0.21611621431222558, "val_accuracy": 0.2, "val_macro_f1": 0.08588235294117648}
{"epoch": 2, "train_loss": 1.690741583279201, "val_loss": 1.616389414469401, "train_accuracy": 0.20285714285714285, "train_macro_f1": 0.2022774444561269, "val_accuracy": 0.22, "val_macro_f1": 0.1538877620013523}
{"epoch": 3, "train_loss": 1.6470955065318515, "val_loss": 1.7034501028060913, "train_accuracy": 0.24857142857142858, "train_macro_f1": 0.24149946062567423, "val_accuracy": 0.2, "val_macro_f1": 0.09212121212121212}
{"epoch": 4, "train_loss": 1.6057263197217668, "val_loss": 1.5464113267262776, "train_accuracy": 0.26571428571428574, "train_macro_f1": 0.2637997787557443, "val_accuracy": 0.2733333333333333, "val_macro_f1": 0.19735872474110847}
{"epoch": 5, "train_loss": 1.4433844767298016, "val_loss": 1.2634836435317993, "train_accuracy": 0.35428571428571426, "train_macr

In [31]:
import json
from pathlib import Path

runs_dir = Path('runs')
run_dirs = sorted([p for p in runs_dir.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
latest_run = run_dirs[-1]
print('Latest run:', latest_run)

metrics = json.loads((latest_run / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2))

Latest run: runs/resnet10_landmark_fusion_20260407_213642
{
  "val_loss": 0.2870551146194339,
  "test_loss": 0.3041713534753459,
  "val_accuracy": 0.9466666666666667,
  "val_macro_f1": 0.9470270716961607,
  "test_accuracy": 0.9333333333333333,
  "test_macro_f1": 0.9337761011635939
}


In [32]:
from pathlib import Path

print('Validation confusion matrix:')
print((latest_run / 'val_confusion_matrix.csv').read_text())

print('Test confusion matrix:')
print((latest_run / 'test_confusion_matrix.csv').read_text())

Validation confusion matrix:
true/pred,G01,G02,G05,G06,G07
G01,29,1,0,0,0
G02,0,30,0,0,0
G05,0,0,27,3,0
G06,0,1,1,28,0
G07,0,2,0,0,28

Test confusion matrix:
true/pred,G01,G02,G05,G06,G07
G01,29,1,0,0,0
G02,0,30,0,0,0
G05,0,0,28,0,2
G06,0,1,0,28,1
G07,0,4,0,1,25



In [33]:
import shutil
destination = RESULTS_PATH / latest_run.name
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree(latest_run, destination)
print('Copied run to Drive:', destination)

Copied run to Drive: /content/drive/MyDrive/dl_project/colab_runs/resnet10_landmark_fusion_20260407_213642
